1. Import

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.model_selection import (
    TimeSeriesSplit,
    cross_val_score,
    GridSearchCV
)

In [2]:
processed_data = pd.read_csv(
    "../data/processed/spy_features.csv",
    index_col=0,
    parse_dates=True
)

2. Define features

In [3]:
FEATURES = [
    "ret_1d",
    "ret_5d",
    "ret_20d",
    "vol_20d",
    "ma_20_dist",
    "zscore_20d",
    "volume_zscore_20d",
    "ret_lag1",
    "ret_lag2",
    "ret_lag3"
]

3. Classification target

In [4]:
data = processed_data[
    FEATURES + ["next_return"]
].dropna().copy()

data["target_up"] = (
    data["next_return"] > 0
).astype(int)

In [5]:
X = data[FEATURES]
y = data["target_up"]

4. Check Class balance

In [6]:
y.value_counts(), y.value_counts(normalize=True)

(target_up
 1    1385
 0    1108
 Name: count, dtype: int64,
 target_up
 1    0.555556
 0    0.444444
 Name: proportion, dtype: float64)

5. Time cut

In [7]:
train = data.loc[: "2022-12-31"]
test = data.loc["2023-01-01":]

X_train = train[FEATURES]
y_train = train["target_up"]

X_test = test[FEATURES]
y_test = test["target_up"]

print(
    X_train.index.min(),
    X_train.index.max()
)

print(
    X_test.index.min(),
    X_test.index.max()
)

2016-02-02 00:00:00 2022-12-30 00:00:00
2023-01-03 00:00:00 2025-12-30 00:00:00


6. Classification baseline

In [8]:
majority_class = y_train.mode()[0]

baseline_pred = np.full(
    len(y_test),
    majority_class
)

In [9]:
baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

baseline_accuracy

0.5739014647137151

7. Logistic Regression

$P(y=1\mid X) = \frac{1}{1+e^{-z}}$

In [10]:
logit_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "logit",
        LogisticRegression(
            max_iter=1000
        )
    )
])

In [11]:
logit_model.fit(
    X_train,
    y_train
)

Pipeline(steps=[('scaler', StandardScaler()),
                ('logit', LogisticRegression(max_iter=1000))])

In [12]:
y_pred_logit = logit_model.predict(
    X_test
)

In [13]:
y_prob_logit = logit_model.predict_proba(
    X_test
)[:, 1]

10. Classification metrics

In [14]:
accuracy = accuracy_score(
    y_test,
    y_pred_logit
)

precision = precision_score(
    y_test,
    y_pred_logit
)

recall = recall_score(
    y_test,
    y_pred_logit
)

f1 = f1_score(
    y_test,
    y_pred_logit
)

roc_auc = roc_auc_score(
    y_test,
    y_prob_logit
)

In [15]:
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC-AUC:", roc_auc)

Accuracy: 0.5552596537949401
Precision: 0.5697841726618705
Recall: 0.9187935034802784
F1: 0.7033747779751333
ROC-AUC: 0.478538283062645


11. Accuracy
$
Accuracy
=
\frac{\text{correct predictions}}
{\text{all predictions}}$

12. $Precision
=
\frac{TP}
{TP+FP}$

13. $Recall
=
\frac{TP}
{TP+FN}$

14. $F1
=
2\frac{
Precision\times Recall
}{
Precision+Recall
}$

confusion matrix

In [16]:
cm = confusion_matrix(
    y_test,
    y_pred_logit
)

cm

array([[ 21, 299],
       [ 35, 396]])

In [17]:
print(
    classification_report(
        y_test,
        y_pred_logit
    )
)

              precision    recall  f1-score   support

           0       0.38      0.07      0.11       320
           1       0.57      0.92      0.70       431

    accuracy                           0.56       751
   macro avg       0.47      0.49      0.41       751
weighted avg       0.49      0.56      0.45       751



17. Random Forest

In [18]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=4,
    random_state=42
)

In [19]:
rf_model.fit(
    X_train,
    y_train
)

RandomForestClassifier(max_depth=4, n_estimators=300, random_state=42)

In [20]:
y_pred_rf = rf_model.predict(
    X_test
)

y_prob_rf = rf_model.predict_proba(
    X_test
)[:, 1]

19. Random Forest metrics

In [21]:
rf_metrics = {
    "Accuracy": accuracy_score(
        y_test,
        y_pred_rf
    ),

    "Precision": precision_score(
        y_test,
        y_pred_rf
    ),

    "Recall": recall_score(
        y_test,
        y_pred_rf
    ),

    "F1": f1_score(
        y_test,
        y_pred_rf
    ),

    "ROC_AUC": roc_auc_score(
        y_test,
        y_prob_rf
    )
}

20. Logistic vs Random Forest

In [22]:
logit_metrics = {
    "Accuracy": accuracy_score(
        y_test,
        y_pred_logit
    ),

    "Precision": precision_score(
        y_test,
        y_pred_logit
    ),

    "Recall": recall_score(
        y_test,
        y_pred_logit
    ),

    "F1": f1_score(
        y_test,
        y_pred_logit
    ),

    "ROC_AUC": roc_auc_score(
        y_test,
        y_prob_logit
    )
}

In [23]:
results = pd.DataFrame({
    "Logistic": logit_metrics,
    "RandomForest": rf_metrics
}).T

results

,Accuracy,Precision,Recall,F1,ROC_AUC
Logistic,0.555260,0.569784,0.918794,0.703375,0.478538
RandomForest,0.560586,0.570827,0.944316,0.711538,0.489211


22. TimeSeriesSplit

In [24]:
tscv = TimeSeriesSplit(
    n_splits=5
)

In [25]:
for fold, (
    train_idx,
    val_idx
) in enumerate(
    tscv.split(X_train),
    start=1
):

    print(
        fold,
        train_idx[0],
        train_idx[-1],
        val_idx[0],
        val_idx[-1]
    )

1 0 291 292 581
2 0 581 582 871
3 0 871 872 1161
4 0 1161 1162 1451
5 0 1451 1452 1741


25. Logistic cross-validation

In [26]:
logit_cv_scores = cross_val_score(
    logit_model,
    X_train,
    y_train,
    cv=tscv,
    scoring="roc_auc"
)

In [27]:
logit_cv_scores

array([0.50642424, 0.44084934, 0.46087899, 0.53094637, 0.51596129])

In [28]:
logit_cv_scores.mean()

np.float64(0.4910120461394504)

In [29]:
logit_cv_scores.std()

np.float64(0.03429008084970024)

28. GridSearchCV

In [30]:
param_grid = {
    "logit__C": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}

In [31]:
("logit", LogisticRegression())

('logit', LogisticRegression())

In [32]:
grid_search = GridSearchCV(
    estimator=logit_model,
    param_grid=param_grid,
    cv=tscv,
    scoring="roc_auc"
)

In [33]:
grid_search.fit(
    X_train,
    y_train
)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('logit',
                                        LogisticRegression(max_iter=1000))]),
             param_grid={'logit__C': [0.001, 0.01, 0.1, 1, 10, 100]},
             scoring='roc_auc')

In [34]:
grid_search.best_params_

{'logit__C': 0.001}

In [35]:
grid_search.best_score_

np.float64(0.505389468924974)

31. final test evaluation

In [36]:
best_logit = grid_search.best_estimator_

In [37]:
y_pred_best = best_logit.predict(
    X_test
)

y_prob_best = best_logit.predict_proba(
    X_test
)[:, 1]

In [38]:
print(
    "Test Accuracy:",
    accuracy_score(
        y_test,
        y_pred_best
    )
)

print(
    "Test ROC-AUC:",
    roc_auc_score(
        y_test,
        y_prob_best
    )
)

Test Accuracy: 0.5765645805592543
Test ROC-AUC: 0.48383120649651973


33. Random Forest can also CV

In [39]:
rf_param_grid = {
    "n_estimators": [
        100,
        300
    ],

    "max_depth": [
        2,
        4,
        6
    ],

    "min_samples_leaf": [
        5,
        20,
        50
    ]
}

In [40]:
rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42
    ),
    param_grid=rf_param_grid,
    cv=tscv,
    scoring="roc_auc",
    n_jobs=-1
)

In [41]:
rf_grid.fit(
    X_train,
    y_train
)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [2, 4, 6],
                         'min_samples_leaf': [5, 20, 50],
                         'n_estimators': [100, 300]},
             scoring='roc_auc')

In [42]:
rf_grid.best_params_

{'max_depth': 2, 'min_samples_leaf': 5, 'n_estimators': 100}

In [43]:
rf_grid.best_score_

np.float64(0.5070657328081857)

34. Random Forest feature importance

In [44]:
best_rf = rf_grid.best_estimator_

In [45]:
feature_importance = pd.Series(
    best_rf.feature_importances_,
    index=FEATURES
).sort_values(
    ascending=False
)

feature_importance

ret_1d               0.177995
ret_5d               0.151362
ret_lag3             0.112248
ma_20_dist           0.106966
volume_zscore_20d    0.101092
zscore_20d           0.078444
ret_lag1             0.072842
ret_20d              0.072819
vol_20d              0.070420
ret_lag2             0.055812
dtype: float64

In [46]:
logit_prob = best_logit.predict_proba(
    X_test
)[:, 1]

In [47]:
best_rf = rf_grid.best_estimator_

rf_prob = best_rf.predict_proba(
    X_test
)[:, 1]

In [48]:
classification_predictions = pd.DataFrame(
    {
        "actual_up": y_test,
        "logit_prob_up": logit_prob,
        "rf_prob_up": rf_prob,
    },
    index=X_test.index
)

In [49]:
classification_predictions.head()

,actual_up,logit_prob_up,rf_prob_up
Date,,,
2023-01-03,1,0.555641,0.556701
2023-01-04,0,0.535563,0.548075
2023-01-05,1,0.558709,0.554219
2023-01-06,0,0.525219,0.504592
2023-01-09,1,0.551944,0.550855


In [50]:
classification_predictions.to_csv(
    "../data/processed/classification_predictions.csv"
)